# Coverage and Interval Length vs IJ Damping (B=200)

This notebook reads `results/metrics_*.json`, extracts coverage metrics at `B_prefix=200`, separates fitted vs oracle nuisance settings, and plots statistics against `ij_damping`.

Included statistics:
- `raw_scores`: pointwise and uniform `fraction_covered`
- `individual_spillover`: pointwise and uniform `fraction_covered`
- `raw_scores`: pointwise and uniform `length_summary.mean`
- `individual_spillover`: pointwise and uniform `length_summary.mean`

In [19]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
JSON_GLOB = "metrics_*.json"
B_TARGET = 200
ALPHA_FILTER = 0.1  # set to None to keep all alphas
TARGETS = ["raw_scores", "individual_spillover"]

# Optional filter on train_fraction.
# Set to None to disable, or set to 1.0 for full-training runs only.
TRAIN_FRACTION_FILTER = 1.0

# Optional filter on pointwise_fraction_covered.
# Set to None to disable, or set as (lower, upper), e.g. (0.90, 0.999).
POINTWISE_FC_RANGE = (0.1, 0.98)
# POINTWISE_FC_RANGE = (0.94, 0.96)

In [20]:
def _prefix_entry_at_B(by_prefix: dict, b_target: int):
    if not by_prefix:
        return None
    # Prefer exact B_prefix match.
    for _, entry in by_prefix.items():
        try:
            if int(entry.get("B_prefix")) == int(b_target):
                return entry
        except Exception:
            continue
    # Fallback: largest available prefix.
    entries = []
    for k, entry in by_prefix.items():
        try:
            b_pref = int(entry.get("B_prefix", k))
            entries.append((b_pref, entry))
        except Exception:
            continue
    if not entries:
        return None
    entries.sort(key=lambda x: x[0])
    return entries[-1][1]


def collect_metrics(results_dir: Path, json_glob: str = "metrics_*.json") -> pd.DataFrame:
    rows = []
    for path in sorted(results_dir.glob(json_glob)):
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        boot = data.get("bootstrap", {})
        nuis = data.get("nuisance", {})
        exp = data.get("experiment", {})
        fit_nuisance = nuis.get("fit_nuisance")
        nuis_label = "fitted" if fit_nuisance is True else ("oracle" if fit_nuisance is False else "unknown")

        train_fraction = exp.get("train_fraction")
        b_file = boot.get("B")
        ij_damping = boot.get("ij_damping")
        coverage = data.get("coverage_by_alpha_k", {})

        for alpha_key, alpha_block in coverage.items():
            for _, k_block in alpha_block.items():
                alpha_val = k_block.get("alpha")
                if ALPHA_FILTER is not None and alpha_val is not None and not np.isclose(float(alpha_val), float(ALPHA_FILTER)):
                    continue

                for split in ("train", "test"):
                    split_block = k_block.get(split, {})
                    for target in TARGETS:
                        target_block = split_block.get(target, {})
                        entry = _prefix_entry_at_B(target_block.get("by_prefix", {}), B_TARGET)
                        if entry is None:
                            continue

                        length_summary = entry.get("length_summary", {})
                        length_pw = length_summary.get("pointwise", {})
                        length_unif = length_summary.get("uniform", {})

                        rows.append(
                            {
                                "file": path.name,
                                "split": split,
                                "target": target,
                                "nuisance": nuis_label,
                                "fit_nuisance": fit_nuisance,
                                "alpha": alpha_val,
                                "B": b_file,
                                "ij_damping": ij_damping,
                                "train_fraction": train_fraction,
                                "B_prefix": entry.get("B_prefix"),
                                "pointwise_fraction_covered": entry.get("pointwise_fraction_covered"),
                                "uniform_fraction_covered": entry.get("uniform_fraction_covered"),
                                "pointwise_length_mean": length_pw.get("mean"),
                                "uniform_length_mean": length_unif.get("mean"),
                            }
                        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    for col in [
        "alpha", "B", "ij_damping", "train_fraction", "B_prefix",
        "pointwise_fraction_covered", "uniform_fraction_covered",
        "pointwise_length_mean", "uniform_length_mean",
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only rows that correspond to B=200 run and B_prefix=200 coverage point.
    df = df[(df["B"] == B_TARGET) & (df["B_prefix"] == B_TARGET)]

    # Optional filter by train_fraction.
    if TRAIN_FRACTION_FILTER is not None:
        df = df[np.isclose(df["train_fraction"], float(TRAIN_FRACTION_FILTER), equal_nan=False)]

    # Optional filter by pointwise fraction covered range.
    if POINTWISE_FC_RANGE is not None:
        lo, hi = POINTWISE_FC_RANGE
        df = df[
            (df["pointwise_fraction_covered"] >= float(lo))
            & (df["pointwise_fraction_covered"] <= float(hi))
        ]

    return df


df = collect_metrics(RESULTS_DIR, JSON_GLOB)


In [21]:
df

,file,split,target,nuisance,fit_nuisance,alpha,B,ij_damping,B_prefix,pointwise_fraction_covered,uniform_fraction_covered,pointwise_length_mean,uniform_length_mean
0,metrics_SBM_n3000_K3_sbm_separate_self_nuis-fi...,test,raw_scores,fitted,True,0.1,200,0.00060,200,0.940774,1.000000,1.096071,2.663165
2,metrics_SBM_n3000_K3_sbm_separate_self_nuis-fi...,test,raw_scores,fitted,True,0.1,200,0.00060,200,0.962794,1.000000,1.938636,4.465531
4,metrics_SBM_n3000_K3_sbm_separate_self_nuis-fi...,test,raw_scores,fitted,True,0.1,200,0.00060,200,0.876614,1.000000,1.541748,3.501700
6,metrics_SBM_n3000_K3_sbm_separate_self_nuis-fi...,test,raw_scores,fitted,True,0.1,200,0.00060,200,0.723614,0.995065,1.075089,2.370477
7,metrics_SBM_n3000_K3_sbm_separate_self_nuis-fi...,test,individual_spillover,fitted,True,0.1,200,0.00060,200,0.954865,1.000000,0.935681,1.821915
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,metrics_SBM_n3000_K3_sbm_separate_self_nuis-or...,test,individual_spillover,oracle,False,0.1,200,0.00036,200,0.675025,0.968907,0.388602,0.823171
196,metrics_SBM_n3000_K3_sbm_separate_self_nuis-or...,test,raw_scores,oracle,False,0.1,200,0.00036,200,0.535308,0.944002,0.637096,1.500124
197,metrics_SBM_n3000_K3_sbm_separate_self_nuis-or...,test,individual_spillover,oracle,False,0.1,200,0.00036,200,0.818455,0.992979,0.551720,1.099068
198,metrics_SBM_n3000_K3_sbm_separate_self_nuis-or...,test,raw_scores,oracle,False,0.1,200,0.00036,200,0.414009,0.841686,0.507404,1.199349


In [22]:
# Aggregate over repeated runs at same (split, target, nuisance, ij_damping)
agg = (
    df.groupby(["split", "target", "nuisance", "ij_damping"], as_index=False)
      .agg(
          n_files=("file", "nunique"),
          pointwise_fraction_covered=("pointwise_fraction_covered", "mean"),
          uniform_fraction_covered=("uniform_fraction_covered", "mean"),
          pointwise_length_mean=("pointwise_length_mean", "mean"),
          uniform_length_mean=("uniform_length_mean", "mean"),
      )
      .sort_values(["split", "target", "nuisance", "ij_damping"])
)

# Filter for 0.925 < pointwise_fraction_covered < 0.975
# agg = agg[(agg["pointwise_fraction_covered"] > 0.945) & (agg["pointwise_fraction_covered"] < 0.955)]

print("aggregated rows after filtering:", len(agg))
display(agg)

aggregated rows after filtering: 4


,split,target,nuisance,ij_damping,n_files,pointwise_fraction_covered,uniform_fraction_covered,pointwise_length_mean,uniform_length_mean
0,test,individual_spillover,fitted,0.00060,12,0.946924,0.999582,0.512337,1.115899
1,test,individual_spillover,oracle,0.00036,50,0.827081,0.987563,0.355000,0.765626
2,test,raw_scores,fitted,0.00060,42,0.905607,0.999833,1.126403,2.671039
3,test,raw_scores,oracle,0.00036,50,0.623554,0.958512,0.450828,1.139331


In [43]:
out_csv = "alpha_0.1.csv"
agg.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

Saved: alpha_0.1.csv


In [ ]:
def plot_stat_vs_damping(agg_df: pd.DataFrame, stat_col: str, split: str, target: str):
    sub = agg_df[(agg_df["split"] == split) & (agg_df["target"] == target)].copy()
    if sub.empty:
        print(f"No data for split={split}, target={target}, stat={stat_col}")
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    for nuis in sorted(sub["nuisance"].dropna().unique()):
        line_df = sub[sub["nuisance"] == nuis].sort_values("ij_damping")
        ax.plot(line_df["ij_damping"], line_df[stat_col], marker="o", label=nuis)

    ax.set_xlabel("ij_damping")
    ax.set_ylabel(stat_col)
    ax.set_title(f"{target} | {split} | {stat_col} vs ij_damping (B=200)")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(title="nuisance")
    plt.tight_layout()
    plt.show()


STAT_COLS = [
    "pointwise_fraction_covered",
    "uniform_fraction_covered",
    "pointwise_length_mean",
    "uniform_length_mean",
]

for split in sorted(agg["split"].dropna().unique()):
    for target in ["raw_scores", "individual_spillover"]:
        for stat_col in STAT_COLS:
            plot_stat_vs_damping(agg, stat_col, split, target)